# Práctica autónoma · Pandas y PostgreSQL
Sin nota. Tiempo sugerido: 90–150 minutos fuera de las clases; S6 es ampliación. Entrega personal sugerida: código, resultados y explicaciones breves. No contiene ejercicios de convolución ni soluciones del trabajo final.

Primero lee las secciones relacionadas de la magistral. Para SQL, carga una vez `sql/00_crear_y_cargar.sql` en tu base de práctica; usa psql en Ubuntu desde VS Code. Los bloques SQL no se ejecutan como celdas Python.


In [1]:
from pathlib import Path
from io import BytesIO
import os, sys, hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option('display.max_columns', 12)
print('Python:', sys.version.split()[0], '| Pandas:', pd.__version__)


Python: 3.12.14 | Pandas: 3.0.1


In [2]:
URL_CSV = 'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv'
datos_originales = pd.read_csv(URL_CSV, na_values=['NA'])
print('Filas y columnas:', datos_originales.shape)
display(datos_originales.head())
# Alternativa offline, tras subir el CSV incluido:
# datos_originales = pd.read_csv('penguins.csv', na_values=['NA'])
# En Jupyter local, leer desde el PC:
# datos_originales = pd.read_csv(r'C:/ruta/penguins.csv', na_values=['NA'])


Filas y columnas: (344, 8)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [3]:
def preparar_tabla(original):
    tabla = original.copy()
    # Es una clave de fila de ESTA copia, no un identificador biológico del animal.
    tabla.insert(0, 'registro_id', np.arange(1, len(tabla) + 1))
    tabla = tabla.rename(columns={
        'species':'especie', 'island':'isla', 'sex':'sexo', 'year':'anio',
        'bill_length_mm':'pico_largo_mm', 'bill_depth_mm':'pico_alto_mm',
        'flipper_length_mm':'aleta_mm', 'body_mass_g':'masa_g'})
    tabla['sexo'] = tabla['sexo'].astype('string').str.strip().str.lower()
    tabla['masa_kg'] = tabla['masa_g'] / 1000.0
    return tabla
datos = preparar_tabla(datos_originales)


## P1 · Selección reproducible
Selecciona Gentoo con masa de al menos 6000 g. Conserva registro_id, isla y masa_g. Ordena por masa descendente y, en empates, registro_id ascendente. Explica por qué el segundo criterio importa.


In [4]:
# Escribe aquí tu solución.
# Añade una celda Markdown con tu interpretación.


## P2 · Ausencias
Cuenta las ausencias de masa y sexo por separado. ¿Cuántas filas perderías al exigir ambas variables? ¿Por qué no se justifica esa eliminación si solo analizas masa? Presenta los conteos y una explicación.


In [5]:
# Escribe aquí tu solución.
# Añade una celda Markdown con tu interpretación.


## P3 · Unidades y copia
Crea una copia con masa en kilogramos sin cambiar datos_originales. Verifica a mano las tres primeras conversiones y su promedio. Explica si cambiar de unidades altera el número de observaciones válidas.


In [6]:
# Escribe aquí tu solución.
# Añade una celda Markdown con tu interpretación.


## P4 · Resumen por isla
Obtén total de filas, masas observadas, media y desviación estándar por isla. Agrega cobertura porcentual de masa. Explica por qué total y observadas son columnas diferentes y por qué no podemos atribuir una diferencia de medias solo a la isla.


In [7]:
# Escribe aquí tu solución.
# Añade una celda Markdown con tu interpretación.


## P5 · Unión controlada
Une a cada registro la media de su isla. Verifica que se conserva el número de filas y calcula la diferencia respecto a esa media. ¿Qué validación detectaría una tabla de medias con una isla duplicada?


In [8]:
# Escribe aquí tu solución.
# Añade una celda Markdown con tu interpretación.


## P6 · Forma y exportación
Construye una tabla de media de masa por especie (filas) y año (columnas). Devuélvela a formato largo. Explica por qué un valor ausente no debe reemplazarse automáticamente por cero. Exporta el formato largo a CSV sin índice y comprueba su lectura. Opcional: guarda otra copia en Drive.


In [9]:
# Escribe aquí tu solución.
# Añade una celda Markdown con tu interpretación.


## S1 · Filtrar y ordenar
Obtén los cinco registros Chinstrap con mayor longitud de aleta. Muestra identificador, isla, aleta y masa; resuelve empates por identificador ascendente. Excluye aletas ausentes explícitamente.

```sql
-- Escribe tu consulta en el psql.
```

Anota aquí tu interpretación y las comprobaciones realizadas.


## S2 · NULL y conteos
Por especie, informa filas totales, masas válidas y masas ausentes. Formula las ausencias como diferencia entre dos conteos. Explica por qué masa_g = NULL no responde la pregunta.

```sql
-- Escribe tu consulta en el psql.
```

Anota aquí tu interpretación y las comprobaciones realizadas.


## S3 · WHERE y HAVING
Entre registros con masa observada, resume por isla y conserva grupos cuya masa media supere 4000 g. Explica cómo cambiaría el significado si filtras cada registro por masa mayor que 4000 antes de agrupar.

```sql
-- Escribe tu consulta en el psql.
```

Anota aquí tu interpretación y las comprobaciones realizadas.


## S4 · Conservar categorías
Usa el catálogo especies para contar registros con masa de al menos 6000 g por especie, incluidas las especies sin tales registros. Decide dónde ubicar el filtro y qué columna contar.

```sql
-- Escribe tu consulta en el psql.
```

Anota aquí tu interpretación y las comprobaciones realizadas.


## S5 · Transacción y restricciones
En una sola ejecución, abre una transacción, crea una tabla temporal ensayo_practica con id como clave primaria y medida positiva, inserta una fila válida, consúltala y revierte. Explica qué rechazarían las restricciones y por qué la tabla no queda disponible después del rollback. No cambies las observaciones originales.

```sql
-- Escribe tu consulta en el psql.
```

Anota aquí tu interpretación y las comprobaciones realizadas.


## S6 · Pipeline local (ampliación sin nota)
Ejecuta `ampliacion/pipeline.sh` siguiendo la guía. Identifica los archivos producidos y explica qué hacen Bash, Python y PostgreSQL. En una copia de los scripts, cambia el resumen de isla a especie; actualiza tanto SQL como las comprobaciones y la figura de Python. Comprueba que si una etapa falla no se genera COMPLETADO.txt. La consulta Psycopg directa es una segunda ampliación, sin conexión con Colab.